In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState
from langgraph.graph import START, END
from llm_factory import LLMFactory
from langchain_core.messages import SystemMessage, trim_messages
from typing import TypedDict  

In [3]:
from config import DATA_DIR
files = [
    # DATA_DIR / "file_02.log",
    # DATA_DIR / 'sh_arp_cache_2020-11-10074812.log',
    # DATA_DIR / 'Microsoft365DefenderEvents.json',
    # DATA_DIR / 'WindowsEvents.json',
    DATA_DIR / "file_01.json",
    DATA_DIR / "file_03.csv",
]

from config import DATA_DIR
files = [
    # DATA_DIR / "file_02.log",
    # DATA_DIR / 'sh_arp_cache_2020-11-10074812.log',
    # DATA_DIR / 'Microsoft365DefenderEvents.json',
    # DATA_DIR / 'WindowsEvents.json',
    DATA_DIR / "file_01.json",
    DATA_DIR / "file_03.csv",
]

print(files)

import json
data = []
with open(files[0], 'r') as f:
    #first_char = f.read(1)
    #f.seek(0)
    print(f)
    data = json.load(f)
print(type(data))
print(f"Total de eventos: {len(data)}")
print(f"Exemplo:\n {data[0]}")

[WindowsPath('C:/davi_tonon/mestrado/data/raw/file_01.json'), WindowsPath('C:/davi_tonon/mestrado/data/raw/file_03.csv')]
<_io.TextIOWrapper name='C:\\davi_tonon\\mestrado\\data\\raw\\file_01.json' mode='r' encoding='cp1252'>
<class 'list'>
Total de eventos: 103
Exemplo:
 {'requestParameters': {'DescribeInstanceTypesRequest': {'NextToken': 'AAIAAUCZLcGdOTmfTz2Vwy7qCVgVq6KNMDDo2s_UFVQdUl8JzmoaM3geYg-eTVO56npOwVkgRcbnccOAIh5xaIntUaFwx3Yzg5z0gJcGwKSvIHr7PoKDSMugzTo27wztP16CU4jRhTPQdzL5kAyA8MMWqgrYKoT5J0xc', 'MaxResults': 100}}, 'userAgent': 'console.ec2.amazonaws.com', 'awsRegion': 'us-east-1', 'eventType': 'AwsApiCall', '@version': '1', 'userIdentity': {'arn': 'arn:aws:iam::123456789123:user/pedro', 'type': 'IAMUser', 'userName': 'pedro', 'sessionContext': {'webIdFederationData': {}, 'sessionIssuer': {}, 'attributes': {'mfaAuthenticated': 'true', 'creationDate': '2020-09-13T17:16:47Z'}}, 'accountId': '123456789123', 'principalId': 'AIDAICAK2CN5MGHIIDIHA', 'accessKeyId': 'ASIA5FLZVX4OI4ZD

In [4]:
SYSTEM_PROMPT = """
You are a cybersecurity analyst specialized in log analysis and MITRE ATT&CK mapping.
You will receive many event of a file. Analyze each event carefully.

You are able to:
- Detect anomalies in logs
- Identify security events
- Map findings to MITRE ATT&CK TTPs

Rules:
- Always base your analysis on evidence
- Do NOT hallucinate
- Be precise and technical
- Always answer in English

### OUTPUT REQUIREMENTS
Provide the results in the following structured format:

**MITRE ATT&CK Mapping**
- **Tactics:**
- **Techniques:**
- **Technique IDs:**



"""

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState  # ← Importante!
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# ❌ NÃO use TypedDict customizado para messages
# ✅ Use MessagesState que já sabe acumular

def analyze_logs(state: MessagesState):
    llm = LLMFactory().get_model()
    llm = ChatOllama(model="llama3.1:8b", temperature=0.1)
    
    # MessagesState já tem 'messages' como chave
    response = llm.invoke(state["messages"])
    
    # Retorna a nova mensagem (MessagesState acumula automaticamente)
    return {"messages": [response]}

builder = StateGraph(MessagesState)  # ← MessagesState aqui
builder.add_node("analyze", analyze_logs)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "sessao-teste"}}

# Teste
result1 = graph.invoke(
    {"messages": [HumanMessage(content=SYSTEM_PROMPT)]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

for i in range(0, len(data), 10):
    chunk = data[i:i+10]
    result2 = graph.invoke(
        {"messages": [HumanMessage(content=f"Analyze this log :\n{chunk}?")]},
        config
    )
    print(f"Resposta {i}:", result2["messages"][-1].content)

# Verifica estado
estado = graph.get_state(config)
print(f"\nTotal de mensagens: {len(estado.values['messages'])}")
for msg in estado.values['messages']:
    print(f"  [{msg.type}] {msg.content[:80]}...")

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: I'm ready to analyze the log events. Please provide the first event for analysis.

Please note that I'll follow the rules you specified, which means I'll only base my analysis on the evidence provided in the logs and avoid making assumptions or hallucinations. My goal is to identify potential security incidents and map them to MITRE ATT&CK TTPs accurately.
LLMFactory: Getting model for provider 'ollama' - metis
Resposta 0: After analyzing the log events, I have identified some potential security incidents and mapped them to MITRE ATT&CK TTPs.

**MITRE ATT&CK Mapping**

* **Tactics:**
	+ Lateral Movement
	+ Privilege Escalation
	+ Command and Control
* **Techniques:**
	+ T1053 (Scheduled Task/Job): The user "pedro" is creating a scheduled task to run the command "DescribeInstanceTypes".
	+ T1055 (Process Injection): The user "pedro" is injecting code into an existing process to execute the command "DescribeInstances".
	

In [6]:
result1 = graph.invoke(
    {"messages": [HumanMessage(content='Provide a final consolidated analysis of all logs')]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)


LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: **Consolidated Analysis**

Based on the provided CloudTrail logs, here's a comprehensive analysis:

**Summary**

The logs indicate that an assumed role (`MordorNginxStack-BankingWAFRole-9S3E0UAE1MM0`) with ARN `arn:aws:sts::123456789123:assumed-role/MordorNginxStack-BankingWAFRole-9S3E0UAE1MM0/i-0317f6c6b66ae9c40` is accessing an S3 bucket (`mordors3stack-s3bucket-llp2yingx64a`) in the `us-east-1` region. The user identity has permission to list objects and retrieve specific objects from the S3 bucket.

**Key Findings**

* **Multiple "ListObjects" operations**: Two consecutive "ListObjects" operations were performed within a short time window (37 seconds), which may indicate a potential security issue, such as a brute-force attack on the S3 bucket.
* **GetObject operation**: A single "GetObject" operation was performed to retrieve an object named `ring.txt` from the S3 bucket. This could be a legitimate operation or an

In [7]:
result1 = graph.invoke(
    {"messages": [HumanMessage(content='Provide a final consolidated analysis of all logs and mapping for Mitre ATT&CK')]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: **Consolidated Analysis**

Based on the provided CloudTrail logs, here's a comprehensive analysis:

**Summary**

The logs indicate that an assumed role (`MordorNginxStack-BankingWAFRole-9S3E0UAE1MM0`) with ARN `arn:aws:sts::123456789123:assumed-role/MordorNginxStack-BankingWAFRole-9S3E0UAE1MM0/i-0317f6c6b66ae9c40` is accessing an S3 bucket (`mordors3stack-s3bucket-llp2yingx64a`) in the `us-east-1` region. The user identity has permission to list objects and retrieve specific objects from the S3 bucket.

**Key Findings**

* **Multiple "ListObjects" operations**: Two consecutive "ListObjects" operations were performed within a short time window (37 seconds), which may indicate a potential security issue, such as a brute-force attack on the S3 bucket.
* **GetObject operation**: A single "GetObject" operation was performed to retrieve an object named `ring.txt` from the S3 bucket. This could be a legitimate operation or an